In [ ]:
# =========================================
# 1. Configuración
# =========================================

CLEAN_FILE = "transactions_clean.parquet"
BUCKET_NAME = "your-gcs-bucket"  # <-- cambia por tu bucket real
BLOB_NAME = "transactions_clean.parquet"

EXPECTED_COLUMNS = [
    "transaction_id",
    "customer_id",
    "transaction_amount",
    "transaction_date",
    "payment_method",
    "product_category",
    "quantity",
    "customer_age",
    "customer_location",
    "device_used",
    "ip_address",
    "shipping_address",
    "billing_address",
    "is_fraudulent",
    "account_age_days",
    "transaction_hour"
]


Saving Fraudulent_E-Commerce_Transaction_Data_2.csv to Fraudulent_E-Commerce_Transaction_Data_2.csv


In [ ]:
# =========================================
# 2. Subir archivo (desde tu computadora)
# =========================================

from google.colab import files

uploaded = files.upload()

# Obtener automáticamente el nombre del archivo
raw_file = next(iter(uploaded.keys()))
print(f"Archivo cargado: {raw_file}")

In [ ]:
# =========================================
# 3. Carga del dataset
# =========================================

import pandas as pd

df = pd.read_csv(
    raw_file,
    sep=",",
    quotechar='"',
    engine="python",
    on_bad_lines="skip"
)

print("Shape inicial:", df.shape)
print("Columnas originales:", df.columns.tolist())
df.head()

(23634, 16)
['Transaction ID', 'Customer ID', 'Transaction Amount', 'Transaction Date', 'Payment Method', 'Product Category', 'Quantity', 'Customer Age', 'Customer Location', 'Device Used', 'IP Address', 'Shipping Address', 'Billing Address', 'Is Fraudulent', 'Account Age Days', 'Transaction Hour']


,Transaction ID,Customer ID,Transaction Amount,Transaction Date,Payment Method,Product Category,Quantity,Customer Age,Customer Location,Device Used,IP Address,Shipping Address,Billing Address,Is Fraudulent,Account Age Days,Transaction Hour
0,c12e07a0-8a06-4c0d-b5cc-04f3af688570,8ca9f102-02a4-4207-ab63-484e83a1bdf0,42.32,2024-03-24 23:42:43,PayPal,electronics,1,40,East Jameshaven,desktop,110.87.246.85,5399 Rachel Stravenue Suite 718\nNorth Blakebu...,5399 Rachel Stravenue Suite 718\nNorth Blakebu...,0,282,23
1,7d187603-7961-4fce-9827-9698e2b6a201,4d158416-caae-4b09-bd5b-15235deb9129,301.34,2024-01-22 00:53:31,credit card,electronics,3,35,Kingstad,tablet,14.73.104.153,"5230 Stephanie Forge\nCollinsbury, PR 81853","5230 Stephanie Forge\nCollinsbury, PR 81853",0,223,0
2,f2c14f9d-92df-4aaf-8931-ceaf4e63ed72,ccae47b8-75c7-4f5a-aa9e-957deced2137,340.32,2024-01-22 08:06:03,debit card,toys & games,5,29,North Ryan,desktop,67.58.94.93,"195 Cole Oval\nPort Larry, IA 58422","4772 David Stravenue Apt. 447\nVelasquezside, ...",0,360,8
3,e9949bfa-194d-486b-84da-9565fca9e5ce,b04960c0-aeee-4907-b1cd-4819016adcef,95.77,2024-01-16 20:34:53,credit card,electronics,5,45,Kaylaville,mobile,202.122.126.216,"7609 Cynthia Square\nWest Brenda, NV 23016","7609 Cynthia Square\nWest Brenda, NV 23016",0,325,20
4,7362837c-7538-434e-8731-0df713f5f26d,de9d6351-b3a7-4bc7-9a55-8f013eb66928,77.45,2024-01-16 15:47:23,credit card,clothing,5,42,North Edwardborough,desktop,96.77.232.76,"2494 Robert Ramp Suite 313\nRobinsonport, AS 5...","2494 Robert Ramp Suite 313\nRobinsonport, AS 5...",0,116,15


In [ ]:
# =========================================
# 4. Estandarización de columnas
# =========================================

df.columns = EXPECTED_COLUMNS

print("Columnas estandarizadas:")
print(df.columns.tolist())


In [ ]:
# =========================================
# 5. Limpieza de datos
# =========================================

# Eliminar duplicados
df = df.drop_duplicates()

# Convertir columnas numéricas
numeric_cols = [
    "transaction_amount",
    "quantity",
    "customer_age",
    "is_fraudulent",
    "account_age_days",
    "transaction_hour"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Eliminar filas con valores críticos nulos
required_cols = [
    "transaction_id",
    "customer_id",
    "transaction_amount",
    "transaction_date",
    "payment_method",
    "product_category",
    "quantity",
    "customer_age",
    "customer_location",
    "device_used",
    "is_fraudulent",
    "account_age_days",
    "transaction_hour"
]

df = df.dropna(subset=required_cols)

# Filtrar valores inválidos
df = df[
    (df["transaction_amount"] > 0) &
    (df["quantity"] > 0) &
    (df["customer_age"] > 0) &
    (df["account_age_days"] >= 0) &
    (df["transaction_hour"].between(0, 23))
].copy()

print("Shape después de limpieza:", df.shape)
df.head()

In [ ]:
# =========================================
# 6. Exportar a Parquet
# =========================================

df.to_parquet(CLEAN_FILE, index=False)
print(f"Archivo exportado: {CLEAN_FILE}")


In [ ]:
# =========================================
# 7. Autenticación en GCP
# =========================================

from google.colab import auth
auth.authenticate_user()

# =========================================
# 8. Instalar cliente de Cloud Storage
# =========================================

!pip install -q google-cloud-storage

# =========================================
# 9. Subir archivo a Cloud Storage
# =========================================

from google.cloud import storage

client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

blob = bucket.blob(BLOB_NAME)
blob.upload_from_filename(CLEAN_FILE)

print(f"Archivo subido a: gs://{BUCKET_NAME}/{BLOB_NAME}")